# 🇪🇹 Amharic Mobile-AGI — Training Notebook
**Soul → Heart → Mind → Body**

Run this on Google Colab (free T4 GPU).
Estimated training time: ~2-4 hours for 50k steps.

In [ ]:
# ── Step 0: Setup ──────────────────────────────────────────
!pip install torch transformers tokenizers datasets sentencepiece onnx onnxruntime Wikipedia-API -q

# Upload your project zip or clone from GitHub
# Option A: Upload zip
# from google.colab import files
# files.upload()  # upload amharic_agi.zip
# !unzip amharic_agi.zip

# Option B: Mount Drive
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/amharic_agi')

In [ ]:
# ── Step 1: Check GPU ──────────────────────────────────────
import torch
print(f'GPU available: {torch.cuda.is_available()}')
print(f'GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB' if torch.cuda.is_available() else '')

In [ ]:
# ── Step 2: Acquire Data ───────────────────────────────────
import sys
sys.path.insert(0, '.')
from main import stage_acquire
stage_acquire()

In [ ]:
# ── Step 3: Clean Data ─────────────────────────────────────
from main import stage_clean
texts = stage_clean()

In [ ]:
# ── Step 4: Train Tokenizer ────────────────────────────────
from main import stage_tokenizer
tokenizer = stage_tokenizer(texts)

In [ ]:
# ── Step 5: Fine-tune Model ────────────────────────────────
# Modify config for Colab memory limits if needed:
from config import CFG
CFG.training.batch_size = 4          # reduce if OOM
CFG.training.max_steps  = 10000      # quick test
CFG.training.fp16       = True       # use mixed precision

from main import stage_finetune
model, tokenizer = stage_finetune(tokenizer)

In [ ]:
# ── Step 6: Compress ───────────────────────────────────────
from main import stage_compress
model_q, tokenizer = stage_compress(model, tokenizer)

In [ ]:
# ── Step 7: Export to Mobile ───────────────────────────────
from main import stage_export
bundle = stage_export(model_q, tokenizer)
print(f'Bundle ready: {bundle}')

In [ ]:
# ── Step 8: Evaluate ───────────────────────────────────────
from main import stage_test
report = stage_test(model, tokenizer)

In [ ]:
# ── Step 9: Download bundle ────────────────────────────────
from google.colab import files
files.download('models/exported/hasab_mobile_bundle.zip')

In [ ]:
# ── Quick Chat Test ────────────────────────────────────────
from body.runtime import MobileRuntime
from config import CFG

runtime = MobileRuntime(CFG.paths.exported)
runtime.load()

prompts = ['ሰላም', 'ኢትዮጵያ ምንድን ናት?', 'አማርኛ ስንት ፊደሎች አሉት?']
for p in prompts:
    response = runtime.chat(p)
    print(f'You:   {p}')
    print(f'Hasab: {response}')
    print()